# 🔍 Phân tích vấn đề TotalCharges

Nhìn vào dữ liệu thực tế để hiểu tại sao `TotalCharges` có khoảng trắng và mối liên hệ với `tenure`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'Dataset shape: {df.shape}')
df.head(3)

---
## Bước 1: TotalCharges có dtype gì?

In [ ]:
print('=== Kiểu dữ liệu của TotalCharges ===')
print(f'dtype: {df["TotalCharges"].dtype}')  # object = string!

print('\n=== 10 giá trị đầu tiên ===')
print(df['TotalCharges'].head(10).tolist())

print('\n=== So sánh với MonthlyCharges (float) ===')
print(f'MonthlyCharges dtype: {df["MonthlyCharges"].dtype}')

---
## Bước 2: Tìm các giá trị không thể chuyển thành số

In [ ]:
# errors='coerce' → giá trị không phải số → NaN
converted = pd.to_numeric(df['TotalCharges'], errors='coerce')

print(f'Tổng số NaN sau khi ép kiểu: {converted.isna().sum()}')

# Tạo mask để lọc ra các dòng lỗi
mask = converted.isna()

print('\n=== Giá trị thực sự của TotalCharges bị lỗi (repr để thấy khoảng trắng) ===')
for val in df.loc[mask, 'TotalCharges'].tolist():
    print(f'  repr: {repr(val)}  |  len: {len(val)}')

---
## Bước 3: In ra 11 dòng bị lỗi — nhìn vào tenure

In [ ]:
print('=== 11 dòng có TotalCharges = khoảng trắng ===')
problem_rows = df[mask][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].copy()
problem_rows['TotalCharges_repr'] = problem_rows['TotalCharges'].apply(repr)
display(problem_rows)

print(f'\n👉 Nhận xét: tenure của TẤT CẢ 11 dòng = {problem_rows["tenure"].unique()}')

---
## Bước 4: Kiểm chứng công thức TotalCharges = MonthlyCharges × tenure

In [ ]:
# Chỉ lấy các dòng bình thường (tenure > 0, TotalCharges là số)
df_normal = df[~mask].copy()
df_normal['TotalCharges'] = pd.to_numeric(df_normal['TotalCharges'])

# Tính TotalCharges theo công thức
df_normal['TotalCharges_calc'] = df_normal['MonthlyCharges'] * df_normal['tenure']

# So sánh
df_normal['difference'] = (df_normal['TotalCharges'] - df_normal['TotalCharges_calc']).abs()

print('=== So sánh TotalCharges thực tế vs công thức ===')
display(df_normal[['tenure', 'MonthlyCharges', 'TotalCharges', 'TotalCharges_calc', 'difference']].head(10))

print(f'\nSai số trung bình: {df_normal["difference"].mean():.2f}$')
print(f'Sai số lớn nhất:   {df_normal["difference"].max():.2f}$')
print('\n💡 Sai số nhỏ do khách hàng có thể đổi gói cước trong quá trình sử dụng')

---
## Bước 5: Visualize mối liên hệ tenure → TotalCharges

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Mối liên hệ: tenure vs TotalCharges', fontsize=14, fontweight='bold')

# Plot 1: Scatter tenure vs TotalCharges
axes[0].scatter(
    df_normal['tenure'], df_normal['TotalCharges'],
    alpha=0.2, s=10, color='#2563EB', label='Dữ liệu thực'
)
# Thêm điểm công thức để so sánh
axes[0].scatter(
    df_normal['tenure'], df_normal['TotalCharges_calc'],
    alpha=0.1, s=10, color='#EF4444', label='Công thức (×MonthlyCharges)'
)
# Đánh dấu 11 điểm bị lỗi (tenure=0)
axes[0].scatter(
    [0]*11, [0]*11,
    color='#F59E0B', s=120, zorder=5,
    marker='*', label='11 dòng bị lỗi (tenure=0)'
)
axes[0].set_xlabel('tenure (tháng)', fontsize=11)
axes[0].set_ylabel('TotalCharges ($)', fontsize=11)
axes[0].set_title('tenure vs TotalCharges', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Plot 2: Phân phối tenure
axes[1].hist(df_normal['tenure'], bins=72, color='#2563EB', alpha=0.7, edgecolor='white')
axes[1].axvline(x=0, color='#F59E0B', linewidth=3, linestyle='--',
                label='tenure=0 (11 dòng lỗi)')
axes[1].set_xlabel('tenure (tháng)', fontsize=11)
axes[1].set_ylabel('Số lượng khách hàng', fontsize=11)
axes[1].set_title('Phân phối tenure', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Bước 6: So sánh 3 cách điền missing values

In [ ]:
# Lấy 11 dòng bị lỗi
df_test = df[mask][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']].copy()

# Tính giá trị thực của TotalCharges theo mỗi cách
df_normal_temp = df[~mask].copy()
df_normal_temp['TotalCharges'] = pd.to_numeric(df_normal_temp['TotalCharges'])
median_val = df_normal_temp['TotalCharges'].median()
mean_val   = df_normal_temp['TotalCharges'].mean()

df_test['Cách 1: Điền median'] = round(median_val, 2)
df_test['Cách 2: Điền mean']   = round(mean_val, 2)
df_test['Cách 3: Điền 0']      = 0.0
df_test['Cách 4: M×tenure ✅'] = df_test['MonthlyCharges'] * df_test['tenure']

print('=== So sánh 4 cách điền missing values ===')
display(df_test[[
    'tenure', 'MonthlyCharges',
    'Cách 1: Điền median',
    'Cách 2: Điền mean',
    'Cách 3: Điền 0',
    'Cách 4: M×tenure ✅'
]].head(5))

print(f'\nGiá trị median = {median_val:.2f}$  →  tenure=0 mà TotalCharges={median_val:.2f}$ ???')
print(f'Giá trị mean   = {mean_val:.2f}$  →  tenure=0 mà TotalCharges={mean_val:.2f}$ ???')
print(f'Điền 0         = 0.00$        →  tenure=0, TotalCharges=0$ ✅ hợp lý')
print(f'MonthlyCharges × tenure = 0.0$  →  tenure=0, TotalCharges=0$ ✅ nhất quán')

---
## Bước 7: Visualize sự vô lý của median vs công thức

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('So sánh các cách điền TotalCharges khi tenure=0', fontsize=13, fontweight='bold')

methods = [
    ('Cách 1: Điền Median\n(SAI về nghiệp vụ)', median_val, '#EF4444'),
    ('Cách 2: Điền Mean\n(SAI về nghiệp vụ)',   mean_val,   '#F59E0B'),
    ('Cách 4: MonthlyCharges × tenure\n(ĐÚNG)', 0.0,        '#10B981'),
]

for ax, (title, fill_val, color) in zip(axes, methods):
    # Scatter của dữ liệu bình thường
    ax.scatter(
        df_normal['tenure'], df_normal['TotalCharges'],
        alpha=0.15, s=8, color='#94A3B8'
    )
    # 11 điểm được điền
    monthly_vals = df[mask]['MonthlyCharges'].values
    fill_vals = monthly_vals * 0 + fill_val if fill_val > 0 else monthly_vals * 0
    ax.scatter(
        [0]*11, fill_vals,
        color=color, s=100, zorder=5,
        edgecolors='black', linewidths=0.5
    )
    ax.set_xlabel('tenure (tháng)', fontsize=10)
    ax.set_ylabel('TotalCharges ($)', fontsize=10)
    ax.set_title(title, fontsize=10, fontweight='bold', color=color)
    ax.grid(True, alpha=0.3)

    if fill_val > 0:
        ax.axhline(y=fill_val, color=color, linestyle='--', alpha=0.5)
        ax.annotate(f'Điền = {fill_val:.0f}$\n(tenure=0 mà charges cao??)',
                    xy=(0, fill_val), xytext=(15, fill_val - 200),
                    fontsize=8, color=color,
                    arrowprops=dict(arrowstyle='->', color=color))

plt.tight_layout()
plt.show()

---
## Bước 8: Áp dụng fix và kiểm tra kết quả

In [ ]:
df_clean = df.copy()

# Bước 1: ép kiểu
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')

print('Trước khi fix:')
print(f'  NaN count: {df_clean["TotalCharges"].isna().sum()}')

# Bước 2: điền theo logic nghiệp vụ
mask2 = df_clean['TotalCharges'].isna()
df_clean.loc[mask2, 'TotalCharges'] = (
    df_clean.loc[mask2, 'MonthlyCharges'] * df_clean.loc[mask2, 'tenure']
)

print('\nSau khi fix:')
print(f'  NaN count: {df_clean["TotalCharges"].isna().sum()}')
print(f'  dtype: {df_clean["TotalCharges"].dtype}')

print('\n=== Kiểm tra 11 dòng đã được fix ===')
display(
    df_clean[df_clean['tenure'] == 0][
        ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']
    ]
)
print('✅ Tất cả đều = 0.0 — hợp lý vì tenure=0!')

---
## Tổng kết

In [ ]:
print('=' * 55)
print('  TỔNG KẾT: Vấn đề TotalCharges')
print('=' * 55)
print()
print('1. PHÁT HIỆN:')
print('   TotalCharges có dtype = object (string)')
print('   11 giá trị là " " (khoảng trắng) → không ép được sang số')
print()
print('2. NGUYÊN NHÂN:')
print('   Tất cả 11 dòng đó đều có tenure = 0')
print('   Khách hàng mới → chưa có hoá đơn → hệ thống ghi " "')
print()
print('3. CÔNG THỨC:')
print('   TotalCharges ≈ MonthlyCharges × tenure')
print('   Khi tenure = 0 → TotalCharges = 0')
print()
print('4. XỬ LÝ:')
print('   pd.to_numeric(..., errors="coerce")  →  " " thành NaN')
print('   df.loc[mask, "TotalCharges"] = MonthlyCharges × tenure = 0')
print()
print('5. TẠI SAO KHÔNG DÙNG MEDIAN/MEAN:')
print(f'   Median = ~1397$ cho khách tenure=0 → VÔ LÝ!')
print('   Luôn hỏi "tại sao missing" trước khi chọn cách điền')
print('=' * 55)